# 03 - Ask for movies 🍿

Time for the **G** in RAG.

```
"a war movie, under 2 hours, with tanks"
   │
   ├─ 1. parse      LLM turns it into filters + search text      (self-query)
   ├─ 2. filter     keep only films ≤ 120 min
   ├─ 3. retrieve   hybrid search + rerank over YOUR list
   ├─ 4. discover   not enough? ask TMDb / YouTube for new titles
   ├─ 5. generate   LLM writes the answer, citing [L1], [N2]...
   └─ 6. check      every citation must exist (cheap hallucination guard)
```

In [1]:
# Make the project importable even if it wasn't installed (uv sync installs it)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

%load_ext autoreload
%autoreload 2

from movierag.config import get_config
cfg = get_config()
print(cfg.summary())

bookmarks     : /home/jp/llms/rag/movies-reels.json
state dir     : /home/jp/.movierag
LLM           : hf:Qwen/Qwen3-4B-Instruct-2507 (4-bit)
embeddings    : BAAI/bge-small-en-v1.5
reranker      : cross-encoder/ms-marco-MiniLM-L6-v2
TMDb key      : yes
YouTube key   : MISSING
watch region  : CH
offline mode  : False


## 1. Load a real LLM, again 🐉

Same as workshop notebook 03, with one difference: your laptop has **8 GB** of VRAM.

`Qwen3-4B-Instruct-2507` has 4 billion parameters. In bf16 (2 bytes each) that's ~8 GB:
too much once the embedding model and the browser take their share.

So we load it in **4-bit** (NF4 quantization via `bitsandbytes`): ~0.5 byte per parameter,
~2.5 GB. Small quality loss, big memory win. (Remember the workshop question:
*where do hundreds of millions of parameters physically go?* Into fewer bits.)

Why not Qwen3.5-4B? It is multimodal and "thinks" by default: slower and harder to
parse. The 2507 Instruct model is text-only and answers directly.

In [2]:
from movierag.app import load_components
from movierag.models import gpu_report

c = load_components()      # catalogue + index + reranker + TMDb + YouTube + LLM
llm, recommender = c.llm, c.recommender
print(gpu_report())

/home/jp/llms/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:14<00:00, 28.36it/s]


NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU: 3.9 GB used / 8.0 GB


In [3]:
print(llm.complete("In one sentence: why do people love submarine movies?").text)

People love submarine movies because they combine suspense, isolation, and the thrill of the unknown in a claustrophobic, high-stakes setting that taps into deep human fears and curiosities.


## 2. Self-querying: from words to filters 🧾

We ask the LLM to fill a **Pydantic schema** (`QueryPlan`). LlamaIndex's `structured_predict`
puts the JSON schema in the prompt and parses the answer back into the object.

Small models make mistakes, so regex rules run alongside and win on hard numbers.

In [4]:
from movierag.query_parser import parse_query

for q in [
    "a war movie, under 2 hours, with tanks",
    "something like Under the Skin but a series",
    "a documentary about the ocean, less than 90 minutes",
    "an 80s or older kung fu movie",
]:
    plan, how = parse_query(q, llm)
    print(f"{q}\n   [{how}] search='{plan.search_text}' | {plan.constraints_text()}\n")

a war movie, under 2 hours, with tanks
   [llm] search='war, tanks, military conflict' | format: film, genres: War, <= 120 min

something like Under the Skin but a series
   [llm] search='body horror, alienation, identity, isolation' | format: series, genres: Horror/Drama, like 'Under the Skin'

a documentary about the ocean, less than 90 minutes
   [llm] search='ocean life, marine ecosystems, conservation efforts' | format: documentary, genres: Documentary, <= 90 min

an 80s or older kung fu movie
   [llm+rules] search='kung fu, martial arts, action, revenge, destiny' | genres: Action/Adventure, >= 1980, <= 1989



## 3. The whole loop 🔁

In [5]:
r = recommender.recommend("a war movie, under 2 hours, with tanks")
r.show()

Q: a war movie, under 2 hours, with tanks
plan (llm): search='war, tanks, military conflict' | format: film, genres: War, <= 120 min

From your list:
  [L1] The Tank (2025) film, 117 min  score=3.355
  [L2] White Tiger (2012) film, 105 min  score=0.084
  [L3] Prisoner of War (2025) film, 113 min  score=-4.511
  [L4] The Silent Service (2023) film, 113 min  score=-6.913
  [L5] See You Up There (2017) film, 113 min  score=-7.707

- [L1] The Tank (2025, 117 min) – A war film featuring a tank crew in WWII, with direct focus on tanks and a mission set in hostile terrain. Fits the war genre and runtime (under 120 min), with explicit tank action and wartime setting.  
- [L2] White Tiger (2012, 105 min) – Set in 1945 during WWII, centers around a Red Army soldier confronting a ghostly-white German Tiger tank. Includes tanks as a key element and fits the war genre and under-120-minute runtime.  
- [L3] Prisoner of War (2025, 113 min) – A WWII war film involving captivity and combat, though it f

### 👀 Look at the prompt

No magic: the LLM sees the question, the constraints and the candidate cards. That's it.
If an answer is wrong, the first question is always: *was the right film in this prompt?*

In [6]:
from movierag.recommender import build_prompt
print(build_prompt(r))

You are a friendly movie-night assistant. The user asked:
"a war movie, under 2 hours, with tanks"

Their constraints: format: film, genres: War, <= 120 min

CANDIDATES FROM THE USER'S SAVED LIST:
[L1] The Tank (film, 2025, 117 min, War/Action/Drama, IMDb 6.5, streaming: Amazon Prime Video)
A German Tiger tank crew is sent on a dangerous mission to rescue the missing officer Paul von Hardenburg from a top-secret bunker behind enemy lines. As they make their way through the lethal no-man's land, they must confront not only the enemy, but also their own fears and inner demons. Fueled by the Wehrmacht's methamphetamine, their mission increasingly becomes a journey into the heart of darkness.

[L2] White Tiger (film, 2012, 105 min, Action/Adventure/Fantasy, IMDb 6.1)
WWII, 1945. After barely surviving a battle with a mysterious, ghostly-white German Tiger tank, Red Army Sergeant Ivan Naydenov becomes obsessed with its destruction.

[L3] Prisoner of War (film, 2025, 113 min, Action/War/Thri

## 4. When your list isn't enough: discovery 🌍

You saved no documentaries. Ask for one and the recommender goes to TMDb:
genre and runtime filters, topic keywords, and *"because you saved X"* recommendations.
New candidates are ranked by the same embedding model + reranker.

In [7]:
recommender.recommend("a documentary about the ocean, less than 90 minutes").show()

Q: a documentary about the ocean, less than 90 minutes
plan (llm): search='ocean life, marine ecosystems, conservation efforts' | format: documentary, genres: Documentary, <= 90 min
  relaxed: no saved title matched genres ['Documentary']; ignored the genre filter

From your list:
New:
  [N1] Ocean with David Attenborough (2025) documentary, 85 min  score=-2.689  https://www.themoviedb.org/movie/1448497
  [N2] The Seafarers (1953) documentary, 29 min  score=-10.088  https://www.themoviedb.org/movie/45314
  [N3] To the Sea (2010) documentary, 73 min  score=-10.386  https://www.themoviedb.org/movie/48414

- [N1] Ocean with David Attenborough (2025, 85 min)  
A visually stunning and scientifically rich documentary that explores the ocean’s vital role in planetary health. With breathtaking cinematography and expert narration, it offers an engaging, educational look at marine ecosystems without exceeding the 90-minute limit.

- [N3] To the Sea (2010, 73 min)  
A heartfelt family documentary

In [8]:
recommender.recommend("something like Under the Skin").show()

Q: something like Under the Skin
plan (llm): search='alien existence, isolation, body horror, existential dread' | genres: Science Fiction/Horror, like 'Under the Skin'

From your list:
  [L1] A Quiet Place Part II (2021) film, 97 min  score=-7.277
  [L2] Hell Hole (2024) film, 92 min  score=-7.826
  [L3] Alien Invasion: S.U.M.1 (2017) film, 95 min  score=-8.787
  [L4] The Abyss (1989) film, 140 min  score=-9.161
  [L5] Vivarium (2019) film, 99 min  score=-9.781

- [L1] A Quiet Place Part II (2021) – Continues the sci-fi horror theme of silent predators and survival under extreme threat, with a strong atmospheric tension and alien-like dangers that echo the body horror and isolation of *Under the Skin*.  
- [L5] Vivarium (2019) – Features a surreal, isolated setting and a disturbing sense of otherness, with a trapped couple forced to survive in a mysterious, non-human environment—aligning with the eerie, body-horror tone of *Under the Skin*.  

No other candidates strongly match the bl

In [9]:
recommender.recommend("a youtube video essay about kung fu cinema, under 30 minutes").show()

Q: a youtube video essay about kung fu cinema, under 30 minutes
plan (llm): search='kung fu, martial arts, cultural heritage, action sequences' | format: youtube, genres: Action/Comedy/Documentary, <= 30 min
  relaxed: no saved title matched genres ['Action', 'Comedy', 'Documentary']; ignored the genre filter

From your list:

None of the available candidates—either from the saved list or new suggestions—match the request for a YouTube video essay about Kung Fu cinema under 30 minutes, with genres aligned to Action/Comedy/Documentary. No suitable titles or descriptions were found that meet all the specified criteria.  

Therefore, no recommendations can be made.

parse 4.6s | retrieve 0.0s | discover 0.0s | generate 4.1s | total 8.8s


## 5. Your turn ✏️

Ask it what you actually want to watch tonight.

In [10]:
recommender.recommend("I have 90 minutes tonight, something gripping").show()

LLM query parsing failed (1 validation error for QueryPlan
kinds
  Input should be a valid array [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/list_type); using rules only


Q: I have 90 minutes tonight, something gripping
plan (rules): search='gripping' | <= 90 min

From your list:
  [L1] Bloodhounds (2023) series, 61 min  score=-9.872
  [L2] Devs (2020) series, 52 min  score=-9.885
  [L3] The Weapon (2012) film, 85 min  score=-9.921
  [L4] Tomb Adventurer (2022) film, 90 min  score=-10.027
  [L5] Subdue the Devil (2022) film, 82 min  score=-10.070

- [L1] Bloodhounds (2023) – A gripping 61-minute drama-action series about two young boxers uniting to fight ruthless foes and protect their loved ones; high-stakes, action-driven, and emotionally intense, perfect for a thrilling 90-minute movie-night.  
- [L3] The Weapon (2012) – An 85-minute drama centered on military weapons testing and tense moral dilemmas; features high tension, real-world stakes, and a strong narrative arc involving danger and secrecy.  
- [L4] Tomb Adventurer (2022) – A full 90-minute action-fantasy film with mythical beasts, high-risk adventures, and a clear quest to escape a deadly la

## 6. Optional: see every step in a trace UI 🔭

```
uv sync --extra tracing
uv run phoenix serve        # in a terminal, then open http://localhost:6006
```

In [ ]:
# from movierag.tracing import enable_tracing
# enable_tracing()
# recommender.recommend("a tense submarine thriller").show()

## 7. Optional: swap the brain 🧠

The LLM sits behind LlamaIndex's `LLM` interface, so changing it is one line.
In `.env`: `LLM_BACKEND=ollama` (local Ollama) or `openai_like` (e.g. the bigger Qwen
on Modal, see `deploy/modal_vllm.py`). Free the GPU first:

In [ ]:
# from movierag.models import free_gpu, load_llm
# del llm, c.recommender.llm; free_gpu()
# c.recommender.llm = load_llm(c.cfg, backend="openai_like")

# ✅ What we just built

A complete RAG loop, where every step is visible and replaceable:

**parse → filter → hybrid retrieve → rerank → discover → generate → check**

The LLM never "knows" your films. It only reads what retrieval puts in front of it.

## Next: is it actually any good? 📏